# Assignment 4: Image Recognition and Classification with R

**R Programming (ODD 2026-27)** | Purvaj Gaonkar | 23102C0083 | BE CMPN-C

Implementation of the project demonstrated in Bharatendra Rai's *Image Recognition
and Classification with R* (Deep Learning with R playlist, lecture 4).

**Objective.** Build and train a neural network in R that classifies images into two
categories, planes and cars, using EBImage for image handling and Keras for the model.

**Before running:** set the runtime to R via **Runtime -> Change runtime type -> R**,
then **Runtime -> Run all**.

In [ ]:
R.version.string

---
## Setup

EBImage is a Bioconductor package, not a CRAN one, so `install.packages()` will not
find it. It also compiles against system libraries for FFTW, TIFF, JPEG and PNG,
which have to be present first. This cell takes a few minutes on a fresh runtime.

In [ ]:
system("apt-get update -qq", intern = FALSE)
system("apt-get install -y libfftw3-dev libtiff-dev libjpeg-dev libpng-dev", intern = FALSE)
cat("system libraries installed\n")

In [ ]:
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
if (!requireNamespace("EBImage", quietly = TRUE)) BiocManager::install("EBImage", ask = FALSE, update = FALSE)
cat("EBImage ready\n")

The tutorial uses the `keras` package. That package is superseded; the current one
is `keras3`. Recent versions resolve their Python backend through
`reticulate::py_require()`, so a separate `install_keras()` call is usually not needed.

In [ ]:
if (!requireNamespace("keras3", quietly = TRUE)) install.packages("keras3")
cat("keras3 ready\n")

In [ ]:
library(EBImage)
library(keras3)

cat("EBImage version:", as.character(packageVersion("EBImage")), "\n")
cat("keras3  version:", as.character(packageVersion("keras3")), "\n")

Smoke test for the backend. If this prints a model summary, Keras is working and
the rest of the notebook will run. If it errors, run `keras3::install_keras()` in a
new cell and then re-run this one.

In [ ]:
smoke <- keras_model_sequential(input_shape = c(4)) |>
  layer_dense(units = 2, activation = "softmax")

summary(smoke)

In [ ]:
dir.create("data/images", recursive = TRUE, showWarnings = FALSE)
dir.create("output", showWarnings = FALSE)
cat("folders ready\n")

---
## Dataset

The tutorial uses twelve personal photographs named `p1.jpg` to `p6.jpg` for planes
and `c1.jpg` to `c6.jpg` for cars.

To keep the project reproducible for anyone who clones the repository, the images are
drawn from **CIFAR-10**, using class 0 (airplane) and class 1 (automobile), and written
out as JPGs with the tutorial's naming scheme. If you would rather use your own twelve
photographs, upload them into `data/images/` with those exact names and this cell will
detect them and skip the generation step.

In [ ]:
expected <- c(paste0("p", 1:6, ".jpg"), paste0("c", 1:6, ".jpg"))
have_own  <- all(file.exists(file.path("data/images", expected)))

if (have_own) {
  cat("Found 12 user-supplied images in data/images/ - using those.\n")
} else {
  cat("Generating 12 images from CIFAR-10 (class 0 = airplane, class 1 = automobile)...\n")

  cifar <- dataset_cifar10()
  x <- cifar$train$x
  y <- as.vector(cifar$train$y)

  plane_idx <- which(y == 0)[1:6]
  car_idx   <- which(y == 1)[1:6]

  write_img <- function(arr, path) {
    img <- Image(aperm(arr / 255, c(2, 1, 3)), colormode = "Color")
    writeImage(img, path, quality = 100)
  }

  for (i in 1:6) write_img(x[plane_idx[i], , , ], file.path("data/images", paste0("p", i, ".jpg")))
  for (i in 1:6) write_img(x[car_idx[i], , , ],   file.path("data/images", paste0("c", i, ".jpg")))

  cat("Wrote 12 images to data/images/\n")
}

list.files("data/images")

---
## Step 1: Read the images

The order matters for everything downstream. Positions 1 to 6 are planes and will be
labelled 0, positions 7 to 12 are cars and will be labelled 1.

In [ ]:
pics <- c('p1.jpg', 'p2.jpg', 'p3.jpg', 'p4.jpg', 'p5.jpg', 'p6.jpg',
          'c1.jpg', 'c2.jpg', 'c3.jpg', 'c4.jpg', 'c5.jpg', 'c6.jpg')

mypic <- list()
for (i in 1:12) {
  mypic[[i]] <- readImage(file.path("data/images", pics[i]))
}

cat("Loaded", length(mypic), "images\n")

## Step 2: Explore the images

In [ ]:
print(mypic[[1]])

`display()` defaults to opening a browser window, which does not exist in Colab.
Passing `method = "raster"` draws the image inline instead.

In [ ]:
display(mypic[[1]], method = "raster")
title("p1.jpg - plane")

In [ ]:
display(mypic[[8]], method = "raster")
title("c2.jpg - car")

In [ ]:
summary(mypic[[1]])

In [ ]:
hist(mypic[[2]], main = "Pixel intensity distribution - p2.jpg")

In [ ]:
str(mypic[[1]])

All twelve images side by side.

In [ ]:
png("output/all_images.png", width = 900, height = 400)
par(mfrow = c(2, 6), mar = c(1, 1, 2, 1))
for (i in 1:12) {
  display(mypic[[i]], method = "raster")
  title(pics[i], cex.main = 1)
}
dev.off()

par(mfrow = c(2, 6), mar = c(1, 1, 2, 1))
for (i in 1:12) {
  display(mypic[[i]], method = "raster")
  title(pics[i], cex.main = 1)
}
par(mfrow = c(1, 1))

---
## Step 3: Resize

Every image is forced to 28 x 28 so that all inputs are the same length.

In [ ]:
for (i in 1:12) {
  mypic[[i]] <- resize(mypic[[i]], 28, 28)
}

str(mypic[[1]])

## Step 4: Reshape

Each 28 x 28 image has 3 colour channels, so flattening gives 28 x 28 x 3 = **2352**
values per image. That number becomes the input width of the network.

In [ ]:
for (i in 1:12) {
  mypic[[i]] <- array_reshape(mypic[[i]], c(28, 28, 3))
}

cat("Length of one flattened image:", length(mypic[[1]]), "\n")

## Step 5: Build the training and test sets

Images 6 and 12, one plane and one car, are held out for testing. The remaining ten
are used for training.

Note that the published script loops only over `7:11`, which would give five training
rows against ten labels. Both ranges are needed.

In [ ]:
trainx <- NULL
for (i in 1:5)  trainx <- rbind(trainx, mypic[[i]])
for (i in 7:11) trainx <- rbind(trainx, mypic[[i]])

testx <- rbind(mypic[[6]], mypic[[12]])

trainy <- c(0, 0, 0, 0, 0, 1, 1, 1, 1, 1)
testy  <- c(0, 1)

str(trainx)
str(testx)

## Step 6: One-hot encode the labels

The output layer has two units, so the labels have to be two columns rather than a
single 0 or 1.

In [ ]:
trainLabels <- to_categorical(trainy)
testLabels  <- to_categorical(testy)

trainLabels

---
## Step 7: Define the model

Three dense layers: 256 and 128 units with ReLU, then 2 units with softmax to produce
a probability for each class.

The tutorial passes `input_shape` inside the first `layer_dense()`. In keras3 the
current form declares it on `keras_model_sequential()` instead, which is what is used
here.

In [ ]:
model <- keras_model_sequential(input_shape = c(2352)) |>
  layer_dense(units = 256, activation = 'relu') |>
  layer_dense(units = 128, activation = 'relu') |>
  layer_dense(units = 2,   activation = 'softmax')

summary(model)

## Step 8: Compile

In [ ]:
model |> compile(
  loss      = 'binary_crossentropy',
  optimizer = optimizer_rmsprop(),
  metrics   = c('accuracy')
)

cat("Model compiled\n")

## Step 9: Train

Thirty epochs with a batch size of 32. With only ten training images and
`validation_split = 0.2`, two images are held back for validation, so the validation
curve is based on a sample of two and should not be read as meaningful.

In [ ]:
history <- model |> fit(
  trainx,
  trainLabels,
  epochs = 30,
  batch_size = 32,
  validation_split = 0.2
)

In [ ]:
plot(history)

In [ ]:
png("output/training_history.png", width = 800, height = 500)
plot(history)
dev.off()
cat("saved output/training_history.png\n")

---
## Step 10: Evaluate on the training data

In [ ]:
train_eval <- model |> evaluate(trainx, trainLabels)
train_eval

### The deprecated call, and its replacement

The tutorial uses `predict_classes()` and `predict_proba()`. Both were **removed in
TensorFlow 2.6** and no longer exist, so the original script errors at this point.

`predict()` returns the class probabilities. `max.col()` gives the column index of the
largest probability in each row, and subtracting 1 converts that index back to the
0/1 label.

In [ ]:
train_prob <- model |> predict(trainx)
train_pred <- max.col(train_prob) - 1

train_pred

### Confusion matrix - training data

In [ ]:
cm_train <- table(Predicted = train_pred, Actual = trainy)
cm_train

In [ ]:
cat("Training accuracy from confusion matrix:",
    round(sum(diag(cm_train)) / sum(cm_train) * 100, 2), "%\n")

Probabilities alongside the predicted and actual labels.

In [ ]:
train_results <- cbind(round(train_prob, 4),
                       Predicted = train_pred,
                       Actual    = trainy)
colnames(train_results)[1:2] <- c("Prob_Plane", "Prob_Car")
train_results

---
## Step 11: Evaluate on the held-out test images

In [ ]:
test_eval <- model |> evaluate(testx, testLabels)
test_eval

In [ ]:
test_prob <- model |> predict(testx)
test_pred <- max.col(test_prob) - 1

test_results <- cbind(round(test_prob, 4),
                      Predicted = test_pred,
                      Actual    = testy)
colnames(test_results)[1:2] <- c("Prob_Plane", "Prob_Car")
rownames(test_results) <- c("p6.jpg", "c6.jpg")
test_results

### Confusion matrix - test data

In [ ]:
cm_test <- table(Predicted = test_pred, Actual = testy)
cm_test

In [ ]:
cat("Test accuracy:", round(sum(diag(cm_test)) / sum(cm_test) * 100, 2), "%\n")

### Confusion matrix as a plot

In [ ]:
plot_cm <- function(cm, title) {
  m <- as.matrix(cm)
  image(1:ncol(m), 1:nrow(m), t(m[nrow(m):1, , drop = FALSE]),
        col = colorRampPalette(c("#f5f7fa", "#2E74B5"))(20),
        axes = FALSE, xlab = "Actual", ylab = "Predicted", main = title)
  axis(1, at = 1:ncol(m), labels = colnames(m))
  axis(2, at = 1:nrow(m), labels = rev(rownames(m)))
  for (i in 1:nrow(m)) for (j in 1:ncol(m))
    text(j, nrow(m) - i + 1, m[i, j], cex = 1.6, font = 2)
  box()
}

png("output/confusion_matrix_train.png", width = 500, height = 450)
plot_cm(cm_train, "Confusion Matrix - Training Data")
dev.off()

plot_cm(cm_train, "Confusion Matrix - Training Data")

In [ ]:
plot_cm(cm_test, "Confusion Matrix - Test Data")

---
## Step 12: Save the outputs

In [ ]:
write.csv(train_results, "output/train_predictions.csv", row.names = FALSE)
write.csv(test_results,  "output/test_predictions.csv",  row.names = TRUE)

metrics <- data.frame(
  Metric = c("Train Loss", "Train Accuracy", "Test Loss", "Test Accuracy", "Epochs",
             "Training Images", "Test Images", "Input Features"),
  Value  = c(round(train_eval$loss, 4), round(train_eval$accuracy, 4),
             round(test_eval$loss, 4),  round(test_eval$accuracy, 4),
             30, nrow(trainx), nrow(testx), ncol(trainx))
)
write.csv(metrics, "output/metrics.csv", row.names = FALSE)

metrics

In [ ]:
save_model(model, "output/plane_car_model.keras")
list.files("output")

To download everything from Colab, zip the folders and use the file browser in the
left sidebar, or run the cell below and download `assignment4_output.zip`.

In [ ]:
system("zip -r assignment4_output.zip output data", intern = FALSE)
cat("created assignment4_output.zip\n")

---
## Interpretation and Limitations

**What the model does.** Each image is flattened into 2352 numbers and passed through
two hidden layers before a two-way softmax. Training accuracy typically reaches 100%
within a few epochs.

**Why that number means very little.** Ten training images against a network with over
600,000 parameters is not learning, it is memorising. There are enough free parameters
to store each training image outright, so perfect training accuracy is the expected
result whether or not the model has learned anything transferable. The honest figure is
the test accuracy, and with a test set of exactly two images it can only ever be 0%,
50% or 100%, so it carries almost no information either.

**The architectural limitation.** Flattening destroys spatial structure. Once the image
is a 2352-long vector, the network has no way to know that two pixels were adjacent, so
it cannot learn edges, corners or shapes. What it can pick up on is the overall
distribution of colour and brightness, which is why the plane and car classes are
separable at all here: sky backgrounds tend to be blue and bright, road backgrounds
tend not to be. A convolutional network preserves that spatial structure through
`layer_conv_2d()` and `layer_max_pooling_2d()`, and is the appropriate architecture for
image data.

**What a real version would need.** Thousands of images per class rather than six, a
proper train/validation/test split, data augmentation, and a convolutional
architecture. The value of this exercise is in the end-to-end workflow, reading and
inspecting image data, preprocessing it into model-ready form, defining, training and
evaluating a network, and validating the results, rather than in the classifier itself.

**One correction to the source material.** The published script calls
`predict_classes()` and `predict_proba()`, both removed in TensorFlow 2.6. They are
replaced here with `predict()` followed by `max.col()`, which is the current idiom.